# Matching and Overlap


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

DESI_PATH = DATA_DIR / "DESI_halos.fits"
BULBUL_PATH = DATA_DIR / "bulbul_clusters.fits"
BALZER_PATH = DATA_DIR / "J_A+A_701_A283" / "balzer_catalog.csv"
XRAY_COMBINED_PATH = DATA_DIR / "xray_combined_catalog.csv"
DESI_OVERLAP_PATH = DATA_DIR / "desi_halos_overlap.csv"
MATCHED_PATH = DATA_DIR / "desi_xray_matched.csv"
UNMATCHED_PATH = DATA_DIR / "desi_xray_unmatched.csv"
DENOMINATOR_PATH = DATA_DIR / "desi_xray_denominator.csv"
MULTIPLE_XRAY_PATH = DATA_DIR / "desi_multiple_xray_candidates.csv"
MULTIPLE_HALO_PATH = DATA_DIR / "desi_multiple_halo_candidates.csv"
CENTRAL_DIAGNOSTIC_PATH = DATA_DIR / "xray_central_diagnostic.csv"


In [ ]:
import pandas as pd
from astropy.table import Table

# -------------------------------
# FITS / TABLE CLEANING UTILITY
# -------------------------------
def fits_to_clean_df(path, hdu=1, one_d_only=True, decode_bytes=True):
    table = Table.read(path, hdu=hdu)

    if one_d_only:
        cols = [c for c in table.colnames if len(table[c].shape) <= 1]
        table = table[cols]

    df = table.to_pandas()

    if decode_bytes:
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].apply(
                    lambda x: x.decode("utf-8")
                    if isinstance(x, (bytes, bytearray))
                    else x
                )

    return df

# -------------------------------
# LOAD INPUT CATALOGS
# -------------------------------
df_DESI = fits_to_clean_df(DESI_PATH)
df_bulbul = fits_to_clean_df(BULBUL_PATH)
df_balzer = pd.read_csv(BALZER_PATH)

# Keep ClusterClass numeric for downstream summaries without excluding catalog rows.
df_balzer["ClusterClass"] = pd.to_numeric(
    df_balzer["ClusterClass"],
    errors="coerce"
)

print("DESI:", df_DESI.shape)
print("Bulbul:", df_bulbul.shape)
print("Balzer:", df_balzer.shape)


In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# STANDARDIZE BULBUL CATALOG
# ==========================================
df_bulbul_match = df_bulbul.copy()
df_bulbul_match["source"] = "bulbul"

# ==========================================
# STANDARDIZE BALZER CATALOG
# ==========================================
df_balzer_match = df_balzer[[
    "IAUName",
    "RAXdeg", "DEXdeg",
    "zBest",
    "ClusterClass",
    "FX300", "CRX300", "ctX300", "LX300",
    "VdispBoot", "lambdaNorm"
]].copy()

# RAXdeg/DEXdeg are the Balzer X-ray peak coordinates.
df_balzer_match = df_balzer_match.rename(columns={
    "RAXdeg": "RA_XFIT",
    "DEXdeg": "DEC_XFIT",
    "zBest": "BEST_Z"
})

df_balzer_match["M500"] = np.nan
df_balzer_match["source"] = "balzer"

# ==========================================
# MERGE CATALOGS
# ==========================================
df_xray = pd.concat(
    [df_bulbul_match, df_balzer_match],
    ignore_index=True,
    sort=False
)

for col in ["RA_XFIT", "DEC_XFIT", "BEST_Z"]:
    df_xray[col] = pd.to_numeric(df_xray[col], errors="coerce")

missing_core = df_xray[["RA_XFIT", "DEC_XFIT", "BEST_Z"]].isna().any(axis=1).sum()

print("Combined X-ray catalog shape:", df_xray.shape)
print(f"Rows missing RA_XFIT/DEC_XFIT/BEST_Z: {missing_core:,}")
print(df_xray[["source", "RA_XFIT", "DEC_XFIT", "BEST_Z"]].head())

# ==========================================
# SAVE OUTPUT
# ==========================================
df_xray.to_csv(
    XRAY_COMBINED_PATH,
    index=False
)

print(f"Saved combined X-ray catalog to: {XRAY_COMBINED_PATH}")


In [ ]:
import pandas as pd
import numpy as np
from astropy.table import Table
import healpy as hp

# ==========================================
# 1. SETUP & LOAD DATA
# ==========================================
print("Loading catalogs...")

# Updated to use the new directories provided
DESI_path = DESI_PATH
xray_path = XRAY_COMBINED_PATH

# Load DESI
DESI_table = Table.read(DESI_path, hdu=1)
df_DESI = DESI_table.to_pandas()

# Filter: STRICT Centrals Only (IS_SAT == False) AND High Mass (M_HALO > 10^13)
df_DESI = df_DESI[(df_DESI['IS_SAT'] == False) & (df_DESI['M_HALO'] > 1e13)].copy().reset_index(drop=True)

# Load X-ray catalog
df_xray = pd.read_csv(xray_path, low_memory=False)

for col in ["RA_XFIT", "DEC_XFIT", "BEST_Z"]:
    df_xray[col] = pd.to_numeric(df_xray[col], errors="coerce")

df_xray = df_xray.dropna(
    subset=["RA_XFIT", "DEC_XFIT", "BEST_Z"]
).copy().reset_index(drop=True)

# ==========================================
# 2. RESTRICTIVE REDSHIFT ENVELOPE
# ==========================================
print("Applying restrictive redshift filter...")

d_z_min, d_z_max = df_DESI['Z'].min(), df_DESI['Z'].max()
x_z_min, x_z_max = df_xray['BEST_Z'].min(), df_xray['BEST_Z'].max()

shared_z_min = max(d_z_min, x_z_min)
shared_z_max = min(d_z_max, x_z_max)

# Filter catalogs BEFORE calculating the footprint
df_DESI = df_DESI[(df_DESI['Z'] >= shared_z_min) & (df_DESI['Z'] <= shared_z_max)].copy().reset_index(drop=True)
df_xray = df_xray[(df_xray['BEST_Z'] >= shared_z_min) & (df_xray['BEST_Z'] <= shared_z_max)].copy().reset_index(drop=True)

# ==========================================
# 3. HEALPIX FOOTPRINT MAP & FILTERING
# ==========================================
print("Mapping data to HEALPix grid...")
nside = 64 # Defines resolution (~0.84 sq deg per pixel)
npix = hp.nside2npix(nside)
pix_area_sqdeg = hp.nside2pixarea(nside, degrees=True)

# Convert RA/DEC to HEALPix Theta/Phi
desi_theta = np.radians(90.0 - df_DESI['DEC'].values)
desi_phi = np.radians(df_DESI['RA'].values)
desi_pixels = hp.ang2pix(nside, desi_theta, desi_phi)

xray_theta = np.radians(90.0 - df_xray['DEC_XFIT'].values)
xray_phi = np.radians(df_xray['RA_XFIT'].values)
xray_pixels = hp.ang2pix(nside, xray_theta, xray_phi)

# Create global binary footprints
desi_footprint = np.zeros(npix, dtype=bool)
desi_footprint[desi_pixels] = True

xray_footprint = np.zeros(npix, dtype=bool)
xray_footprint[xray_pixels] = True

# Calculate the precise overlap
overlap_footprint = desi_footprint & xray_footprint

# Filter data strictly to objects that land in the overlapping pixels
df_DESI_overlap = df_DESI[overlap_footprint[desi_pixels]].copy().reset_index(drop=True)
df_xray_overlap = df_xray[overlap_footprint[xray_pixels]].copy().reset_index(drop=True)

# Calculate Area Stats
area_desi = np.sum(desi_footprint) * pix_area_sqdeg
area_xray = np.sum(xray_footprint) * pix_area_sqdeg
area_overlap = np.sum(overlap_footprint) * pix_area_sqdeg

# ==========================================
# 4. PRINT CONSOLE STATISTICS
# ==========================================
print("\n" + "="*50)
print(" HEALPIX 3D OVERLAP SUMMARY")
print("="*50)
print(f"Redshift Envelope: {shared_z_min:.4f} to {shared_z_max:.4f}")
print(f"Overlap Area: ~{area_overlap:,.0f} sq deg")
print(f"DESI Centrals in Overlap: {len(df_DESI_overlap):,}")
print(f"X-ray Sources in Overlap: {len(df_xray_overlap):,}")
print("="*50 + "\n")

# ==========================================
# 5. SAVE DESI OVERLAP DATA
# ==========================================
output_path = DESI_OVERLAP_PATH

df_DESI_overlap.to_csv(output_path, index=False)

print(f"Saved DESI overlap halos to: {output_path}")


In [ ]:
import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord, search_around_sky
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.constants import G

# -------------------------------
# Paths
# -------------------------------
desi_path = DESI_OVERLAP_PATH
xray_path = XRAY_COMBINED_PATH
out_dir = DATA_DIR

# -------------------------------
# Cosmology
# -------------------------------
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# -------------------------------
# 1. Load DESI halos
# -------------------------------
df_DESI = pd.read_csv(desi_path).copy().reset_index(drop=True)

print(f"Overlap DESI halos (denominator): {len(df_DESI):,}")

# -------------------------------
# 2. Load X-ray catalog
# -------------------------------
df_xray = pd.read_csv(xray_path, low_memory=False)

for col in ["RA_XFIT", "DEC_XFIT", "BEST_Z"]:
    df_xray[col] = pd.to_numeric(df_xray[col], errors="coerce")

df_xray = (
    df_xray
    .dropna(subset=["RA_XFIT", "DEC_XFIT", "BEST_Z"])
    .copy()
    .reset_index(drop=True)
)

print(f"Usable X-ray sources: {len(df_xray):,}")

# -------------------------------
# 3. Compute R200c
# -------------------------------
M_halo = df_DESI["M_HALO"].values * u.Msun
z_halo = df_DESI["Z"].values

Hz = cosmo.H(z_halo).to(1 / u.s)
rho_c = 3 * Hz**2 / (8 * np.pi * G)

R200 = (
    (3 * M_halo.to(u.kg))
    / (4 * np.pi * 200 * rho_c)
) ** (1 / 3)

R200_kpc = R200.to(u.kpc).value

DA = cosmo.angular_diameter_distance(z_halo).to(u.kpc).value

theta_R200 = (
    (R200_kpc / DA) * u.rad
).to(u.arcmin)

df_DESI["R200_kpc"] = R200_kpc
df_DESI["theta_R200_arcmin"] = theta_R200.value

# -------------------------------
# 4. Coordinates
# -------------------------------
coords_DESI = SkyCoord(
    ra=df_DESI["RA"].values * u.deg,
    dec=df_DESI["DEC"].values * u.deg
)

coords_xray = SkyCoord(
    ra=df_xray["RA_XFIT"].values * u.deg,
    dec=df_xray["DEC_XFIT"].values * u.deg
)

# -------------------------------
# 5. Pre-search
# -------------------------------
idx_DESI, idx_xray, d2d, _ = search_around_sky(
    coords_DESI,
    coords_xray,
    20 * u.arcmin
)

# -------------------------------
# 6. Cuts
# -------------------------------
within_match_radius = (
    d2d < 1.5 * theta_R200[idx_DESI]
)

delta_z = np.abs(
    df_DESI["Z"].values[idx_DESI]
    - df_xray["BEST_Z"].values[idx_xray]
)

z_cut = delta_z < 0.005

mask = within_match_radius & z_cut

# -------------------------------
# 7. Candidate table
# -------------------------------
candidates = pd.DataFrame({
    "DESI_idx": idx_DESI[mask],
    "xray_idx": idx_xray[mask],
    "sep_arcmin": d2d[mask].to(u.arcmin).value,
    "delta_z": delta_z[mask],
    "R200_arcmin": theta_R200[idx_DESI[mask]].value
})

candidates["offset_r200c"] = (
    candidates["sep_arcmin"]
    / candidates["R200_arcmin"]
)

print(f"Candidate pairs: {len(candidates):,}")

# =====================================================
# TYPE 1 AMBIGUITY
# One DESI halo -> multiple X-ray candidates
# =====================================================

xray_counts_per_halo = (
    candidates
    .groupby("DESI_idx")
    .size()
    .rename("n_xray_candidates")
)

candidates = candidates.merge(
    xray_counts_per_halo,
    left_on="DESI_idx",
    right_index=True
)

candidates["multiple_xray_candidates"] = (
    candidates["n_xray_candidates"] > 1
)

type1_ambiguous = (
    candidates[
        candidates["multiple_xray_candidates"]
    ]
    .copy()
)

# -------------------------------
# 8. Resolve matches
# Keep smallest offset
# -------------------------------
best_matches = (
    candidates
    .sort_values(
        ["DESI_idx", "offset_r200c"]
    )
    .drop_duplicates(
        subset="DESI_idx",
        keep="first"
    )
    .reset_index(drop=True)
)

# =====================================================
# TYPE 2 AMBIGUITY
# Multiple DESI halos -> same X-ray source
# =====================================================

halo_counts_per_xray = (
    best_matches
    .groupby("xray_idx")
    .size()
    .rename("n_halo_candidates")
)

best_matches = best_matches.merge(
    halo_counts_per_xray,
    left_on="xray_idx",
    right_index=True
)

best_matches["multiple_halo_candidates"] = (
    best_matches["n_halo_candidates"] > 1
)

type2_ambiguous = (
    best_matches[
        best_matches["multiple_halo_candidates"]
    ]
    .copy()
)

# -------------------------------
# 9. Build matched dataset
# -------------------------------
df_DESI_pref = df_DESI.add_prefix("DESI_")
df_xray_pref = df_xray.add_prefix("xray_")

matched_full = (
    best_matches
    .merge(
        df_DESI_pref,
        left_on="DESI_idx",
        right_index=True
    )
    .merge(
        df_xray_pref,
        left_on="xray_idx",
        right_index=True
    )
)

# -------------------------------
# 10. Unmatched
# -------------------------------
matched_ids = set(best_matches["DESI_idx"])

df_unmatched = (
    df_DESI.loc[
        ~df_DESI.index.isin(matched_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

# -------------------------------
# 11. Denominator
# -------------------------------
df_denominator = df_DESI.copy()

# -------------------------------
# 12. Save
# -------------------------------
matched_full.to_csv(
    MATCHED_PATH,
    index=False
)

df_unmatched.to_csv(
    UNMATCHED_PATH,
    index=False
)

df_denominator.to_csv(
    DENOMINATOR_PATH,
    index=False
)

type1_ambiguous.to_csv(
    MULTIPLE_XRAY_PATH,
    index=False
)

type2_ambiguous.to_csv(
    MULTIPLE_HALO_PATH,
    index=False
)

# -------------------------------
# Summary
# -------------------------------
print("\nSaved:")
print("- matched")
print("- unmatched")
print("- denominator")
print("- desi_multiple_xray_candidates")
print("- desi_multiple_halo_candidates")

print("\nAmbiguity summary:")
print(
    f"Halos with multiple X-ray candidates: "
    f"{type1_ambiguous['DESI_idx'].nunique():,}"
)

print(
    f"X-ray sources with multiple halo matches: "
    f"{type2_ambiguous['xray_idx'].nunique():,}"
)


In [ ]:
import pandas as pd
import numpy as np
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u
from IPython.display import display

# ==========================================
# LOAD DATA
# ==========================================
matched = pd.read_csv(MATCHED_PATH)

# ==========================================
# FILTERS
# ==========================================
# Remove ambiguity cases
matched = matched[
    (~matched["multiple_xray_candidates"]) &
    (~matched["multiple_halo_candidates"])
].copy()

print(f"Clean matched systems: {len(matched):,}")

desi = Table.read(DESI_PATH, hdu=1)
desi = desi.to_pandas()

# ==========================================
# OUTPUT STORAGE
# ==========================================
results = []
failed_central_match = 0

# ==========================================
# LOOP OVER MATCHED HALOS
# ==========================================
for _, row in matched.iterrows():

    igrp = row["DESI_IGRP"]

    central_ra = row["DESI_RA"]
    central_dec = row["DESI_DEC"]

    xray_ra = row["xray_RA_XFIT"]
    xray_dec = row["xray_DEC_XFIT"]

    # -------------------------
    # get all halo members
    # -------------------------
    members = desi[
        desi["IGRP"] == igrp
    ].copy()

    if len(members) == 0:
        continue

    # -------------------------
    # locate designated central
    # -------------------------
    central = members[
        (members["RA"] == central_ra) &
        (members["DEC"] == central_dec)
    ]

    if len(central) != 1:
        failed_central_match += 1
        continue

    central_index = central.index[0]

    # -------------------------
    # compute member separations
    # -------------------------
    member_coords = SkyCoord(
        ra=members["RA"].values * u.deg,
        dec=members["DEC"].values * u.deg
    )

    xray_coord = SkyCoord(
        ra=xray_ra * u.deg,
        dec=xray_dec * u.deg
    )

    sep = xray_coord.separation(
        member_coords
    ).arcsec

    members["sep_arcsec"] = sep

    # -------------------------
    # nearest galaxy
    # -------------------------
    closest_index = members[
        "sep_arcsec"
    ].idxmin()

    closest_is_central = (
        closest_index == central_index
    )

    results.append({
        "DESI_IGRP": igrp,
        "xray_idx": row["xray_idx"],
        "n_members": len(members),
        "central_sep_arcsec": members.loc[
            central_index,
            "sep_arcsec"
        ],
        "closest_sep_arcsec": members.loc[
            closest_index,
            "sep_arcsec"
        ],
        "closest_is_central": closest_is_central
    })

# ==========================================
# RESULTS TABLE
# ==========================================
results = pd.DataFrame(results)

results.to_csv(
    CENTRAL_DIAGNOSTIC_PATH,
    index=False
)

# ==========================================
# SUMMARY
# ==========================================
n_total = len(results)

n_central = (
    results["closest_is_central"]
).sum()

n_satellite = (
    ~results["closest_is_central"]
).sum()

print("=" * 50)
print("X-RAY / CENTRAL DIAGNOSTIC")
print("=" * 50)

print(
    f"Matched halos checked: "
    f"{n_total:,}"
)

print(
    f"Central lookup failures: "
    f"{failed_central_match:,}"
)

print()

print(
    f"Closest galaxy IS central: "
    f"{n_central:,}"
)

print(
    f"Closest galaxy is satellite: "
    f"{n_satellite:,}"
)

satellite_fraction = 100 * n_satellite / n_total if n_total > 0 else np.nan

print(
    f"Satellite fraction: "
    f"{satellite_fraction:.2f}%"
)

print()

print("=" * 50)
print("EXAMPLE FAILURES")
print("=" * 50)

display(
    results[
        ~results["closest_is_central"]
    ][[
        "DESI_IGRP",
        "xray_idx",
        "n_members",
        "central_sep_arcsec",
        "closest_sep_arcsec"
    ]]
    .head(10)
)
